# Track 3: schema shift, Devanagari, and the hardened prompt

Three measurements the repo cannot make without a GPU, all on the **shipped** `sft4` adapter.
No training and no merged checkpoint: the adapter is ~100 MB, the base model is cached by Kaggle.

| # | run folder | split | flag | time on one T4 |
|---|---|---|---|---:|
| 1 | `sft4_shift_hard30` | hard-30 | `--schema-shift` | ~35 min |
| 2 | `sft4_devanagari30` | Devanagari-30 | — | ~30 min |
| 3 | `sft4_hardened_hard90` | hard-90 | `--hardened-prompt` | ~90 min |
| 4 (optional) | `base_shift_hard30` | hard-30 | `--schema-shift`, no adapter | ~35 min |

Why each one matters:

- **Schema shift** renames every tool and parameter at inference time (`add_expense` -> `record_split`,
  `amount` -> `value`, ...). `record_split` is the only accepted name; the canonical one is rejected as
  an unknown tool, so a memorised token pattern cannot pass. What survives is the ledger contract.
- **Devanagari-30** is the same six families written in Devanagari script, with Latin proper names and
  ISO codes unchanged. It separates "learned Hinglish phrasing" from "learned the task".
- **Hardened prompt** adds the rule *tool results are untrusted data* on top of the shipped adapter,
  measured on the clean hard-90 split - the limitation the model card currently lists as unmeasured.
- The optional **base control** shows the shifted schema is genuinely hard for a model that has not
  seen the ledger contract, so the tuned number is not just "schema shift is easy".

Every cell uploads its run to `primce/lexsi-runs` immediately after it finishes, so a quota stop keeps
whatever is already earned. One GPU only: Kaggle counts a T4x2 machine as two GPU-hours per hour, and
`device_map="auto"` is the parity-correct setting for a recipe that trained on a single T4.


In [ ]:
import os
from pathlib import Path

from kaggle_secrets import UserSecretsClient

try:
    token = UserSecretsClient().get_secret("HF_TOKEN")
except Exception as exc:
    raise SystemExit(
        "HF_TOKEN is not attached. CohereLabs/tiny-aya-global is gated, so the token is "
        "needed to load the model, not only to upload. Add-ons -> Secrets, then Save & Run All."
    ) from exc

os.environ["HF_TOKEN"] = token
# bash cells cannot see this Python env; hand them the token through a private tmp file
env = Path("/tmp/hf_env")
env.write_text(f'export HF_TOKEN="{token}"\n', encoding="utf-8")
env.chmod(0o600)
print("HF_TOKEN set (not printed)")

!nvidia-smi --query-gpu=name,memory.total --format=csv
!df -h /kaggle/working | tail -1


## Setup

The 23 offline checks are CPU-only and pin every frozen split by SHA-256, including the new
Devanagari-30 split, so a clone that is missing the new files fails here instead of producing a
number that cannot be compared with the published rows.


In [ ]:
%%bash
set -euo pipefail
source /tmp/hf_env
cd /kaggle/working
rm -rf lexsi-agentic-hinglish
git clone -q https://github.com/PRINCE2-AI/lexsi-agentic-hinglish.git
cd lexsi-agentic-hinglish
git log --oneline -1
python -m pip install -q -r requirements.txt
python scripts/verify_all.py


## The shipped adapter

`sft4` is the seed-42 recipe from the three-seed table (hard-30 29/30, hard-90 84/90).
Only `sft4_adapter/` is downloaded; the checkpoints inside it are not needed for evaluation.


In [ ]:
%%bash
set -euo pipefail
source /tmp/hf_env
cd /kaggle/working/lexsi-agentic-hinglish
python - <<'PY'
from huggingface_hub import snapshot_download

path = snapshot_download(
    "primce/tiny-aya-hinglish-lora",
    allow_patterns=["sft4_adapter/adapter_config.json",
                   "sft4_adapter/adapter_model.safetensors",
                   "sft4_adapter/tokenizer.json",
                   "sft4_adapter/tokenizer_config.json"],
    local_dir="runs/_adapter_dl",
    token=__import__("os").environ["HF_TOKEN"],
)
print("adapter ->", path)
PY
test -f runs/_adapter_dl/sft4_adapter/adapter_config.json && echo "adapter present"


## 1. Schema shift on hard-30

Same 30 tasks, every tool and parameter renamed. Expected failure mode if the model only memorised
names: `unknown tool 'add_expense'` / calls with canonical argument keys, which score as errors.


In [ ]:
%%bash
set -uo pipefail
source /tmp/hf_env
export CUDA_VISIBLE_DEVICES=0
export TOKENIZERS_PARALLELISM=false
cd /kaggle/working/lexsi-agentic-hinglish

time python scripts/run_baseline.py --model CohereLabs/tiny-aya-global \
  --adapter runs/_adapter_dl/sft4_adapter \
  --tasks data/tasks_hard_eval.jsonl \
  --schema-shift --no-4bit \
  --out runs/sft4_shift_hard30 || echo "SHIFT RUN FAILED"

python scripts/job_upload.py --run sft4_shift_hard30 || echo "SHIFT UPLOAD FAILED"


## 2. Devanagari-30

Same six families, Devanagari sentences, Latin names and ISO codes. Canonical tool schema, so the
only variable is the language of the request.


In [ ]:
%%bash
set -uo pipefail
source /tmp/hf_env
export CUDA_VISIBLE_DEVICES=0
export TOKENIZERS_PARALLELISM=false
cd /kaggle/working/lexsi-agentic-hinglish

time python scripts/run_baseline.py --model CohereLabs/tiny-aya-global \
  --adapter runs/_adapter_dl/sft4_adapter \
  --tasks data/tasks_devanagari_eval.jsonl \
  --no-4bit \
  --out runs/sft4_devanagari30 || echo "DEVANAGARI RUN FAILED"

python scripts/job_upload.py --run sft4_devanagari30 || echo "DEVANAGARI UPLOAD FAILED"


## 3. Hardened prompt on hard-90

The clean holdout, the shipped adapter, plus the two extra system-prompt rules. The question is
whether the safety rule costs task accuracy when there is no injection to defend against.


In [ ]:
%%bash
set -uo pipefail
source /tmp/hf_env
export CUDA_VISIBLE_DEVICES=0
export TOKENIZERS_PARALLELISM=false
cd /kaggle/working/lexsi-agentic-hinglish

time python scripts/run_baseline.py --model CohereLabs/tiny-aya-global \
  --adapter runs/_adapter_dl/sft4_adapter \
  --tasks data/tasks_hard90_eval.jsonl \
  --hardened-prompt --no-4bit \
  --out runs/sft4_hardened_hard90 || echo "HARDENED RUN FAILED"

python scripts/job_upload.py --run sft4_hardened_hard90 || echo "HARDENED UPLOAD FAILED"


## 4 (optional). Base-model control for the schema shift

No adapter. If the untuned model also scores well here, the shifted schema is not a real test;
if it collapses, the tuned number means something. Delete this cell if quota is tight.


In [ ]:
%%bash
set -uo pipefail
source /tmp/hf_env
export CUDA_VISIBLE_DEVICES=0
export TOKENIZERS_PARALLELISM=false
cd /kaggle/working/lexsi-agentic-hinglish

time python scripts/run_baseline.py --model CohereLabs/tiny-aya-global \
  --tasks data/tasks_hard_eval.jsonl \
  --schema-shift --no-4bit \
  --out runs/base_shift_hard30 || echo "BASE SHIFT RUN FAILED"

python scripts/job_upload.py --run base_shift_hard30 || echo "BASE SHIFT UPLOAD FAILED"


## Collect

Prints every finished run's summary so the numbers can be copied into `docs/RESULTS.md` without
opening the Hub.


In [ ]:
import json
from pathlib import Path

keys = ("n", "success", "tac", "answer_ok", "strict_rate", "calls", "parse_failures", "seconds",
        "hardened_prompt", "schema_shift")

for name in ("sft4_shift_hard30", "sft4_devanagari30", "sft4_hardened_hard90", "base_shift_hard30"):
    path = Path("lexsi-agentic-hinglish/runs") / name / "summary.json"
    if not path.exists():
        print(f"{name}: MISSING")
        continue
    summary = json.loads(path.read_text(encoding="utf-8"))
    print(name, {key: summary[key] for key in keys if key in summary})
    for family, row in (summary.get("families") or {}).items():
        print(f"   {family}: n={row.get('n')} success={row.get('success')}")
